# R Hypothesis Testing and Nonparametric Methods

**Tier 0 -- Computational Foundations | Module 8, Notebook 1**

---

## Overview

This notebook covers the core nonparametric and exact hypothesis tests used in biological and medical research. These methods make minimal distributional assumptions and are therefore essential when sample sizes are small, distributions are skewed, or normality cannot be verified.

### Learning Objectives

By the end of this notebook you will be able to:
- Apply R's `d/p/q/r` distribution function convention to any distribution
- Run exact and asymptotic binomial tests and interpret p-values
- Perform the sign test, Wilcoxon signed-rank test, and Wilcoxon rank-sum / Mann-Whitney U test in R
- Compute Hodges-Lehmann estimates and robust confidence intervals
- Apply the Kruskal-Wallis test for multi-group comparisons and Dunn post-hoc tests
- Construct power functions and determine required sample sizes

## Why this notebook matters

R provides exact implementations of statistical tests that Python's scipy only approximates for small samples. When you have 5–10 biological replicates — common in bench biology — exact tests matter. Bioconductor packages (DESeq2, edgeR) are built on R's testing infrastructure. This notebook teaches you to apply the right test to the right biological question in R, interpret the output, and recognize when the assumptions of a parametric test are violated.


## How to use this notebook
1. All cells use the R kernel (IRkernel). Ensure R is installed with base stats package (included by default).
2. Run each section sequentially — later sections reference datasets created earlier.
3. For each test, note: (a) the null hypothesis being tested, (b) the assumptions, and (c) what the output numbers mean.
4. When comparing parametric and non-parametric tests on the same data, think about which is more appropriate for that data shape.


## Common stumbling points

- **`wilcox.test` vs. Mann-Whitney**: In R, `wilcox.test` with two unpaired samples performs the Mann-Whitney U test. With `paired=TRUE`, it performs the Wilcoxon signed-rank test. The function name is confusing — know which one you need.
- **`p.adjust` method names**: Use `"BH"` for Benjamini-Hochberg FDR, `"bonferroni"` for Bonferroni. Passing `"fdr"` will not work — R uses `"BH"` as the method name.
- **`t.test` default is two-sided**: Use `alternative="greater"` or `alternative="less"` for one-sided tests. In bioinformatics, you should almost always use two-sided unless you have a pre-specified directional hypothesis.
- **ANOVA tells you groups differ, not which ones**: A significant ANOVA F-test only tells you that *at least one* group is different. Use post-hoc tests (TukeyHSD, pairwise.t.test with correction) to find which pairs differ.


---

## 1. R's Distribution Functions

R was built for statistical computing. One of its most practical features is a **consistent naming convention** for distribution functions: every distribution is accessible through four functions that share a common name prefix. Once you learn this convention for one distribution, you know it for all of them.


### 1.1 The d/p/q/r Distribution Function Convention

Every probability distribution in R is accessed through four functions sharing a common prefix:

| Prefix | Returns | Example |
|--------|---------|--------|
| `d` | Probability density (or mass) at *x* | `dnorm(x, mean, sd)` |
| `p` | Cumulative probability P(X ≤ x) | `pnorm(q, mean, sd)` |
| `q` | Quantile — inverse CDF | `qnorm(p, mean, sd)` |
| `r` | Random sample | `rnorm(n, mean, sd)` |

The suffix names the distribution: `norm`, `binom`, `t`, `chisq`, `f`, `exp`, `pois`, `unif`, `nbinom`.

The `lower.tail` argument controls which tail is used:
- `lower.tail = TRUE` (default): P(X ≤ x)
- `lower.tail = FALSE`: P(X > x)

In [ ]:
# P(X > 54) for Binomial(100, 0.25) — strictly greater than 54
pbinom(54, size = 100, prob = 0.25, lower.tail = FALSE)

# P(X > 69) for Binomial(100, 0.85)
pbinom(69, size = 100, prob = 0.85, lower.tail = FALSE)

In [ ]:
# Exponential distribution: P(X <= 12) where mean = 80 min
# rate = lambda = 1 / mean
pexp(12, rate = 1/80)

# Normal distribution: P(X > 252) where mean = 262.5, sd = 12
sigma <- 12
mu    <- 262.5
pnorm(252, mu, sigma, lower.tail = FALSE)

In [ ]:
# Sample size estimation with normal quantile
# How many samples needed so that the 99.5% CI for the difference of two means
# has half-width <= 0.5, given sigma = 12?
sigma <- 12
n_2 <- (sigma / 0.5 * qnorm(0.995, mean = 0, sd = 1))^2
ceiling(n_2)   # ceiling() rounds up to the nearest integer

---

## 2. Exact Binomial Test

The exact binomial test evaluates whether the probability of "success" in a sequence of Bernoulli trials equals a specified value $p_0$.

**Hypotheses:**
- $H_0$: $p = p_0$
- $H_1$: $p > p_0$ (right-sided), $p < p_0$ (left-sided), or $p \neq p_0$ (two-sided)

**Test statistic:** $B = \sum_{i=1}^n \mathbb{1}[X_i = \text{success}]$ — total number of successes.

Under $H_0$, $B \sim \text{Binomial}(n, p_0)$.  
The p-value is computed exactly from the binomial CDF.

In [ ]:
# Example: In a survey of 1002 patients, 701 preferred treatment A over B.
# Test H0: p = 0.5 against H1: p > 0.5 (more patients prefer A).
n   <- 1002
b   <- 701
p_0 <- 0.5

my_test <- binom.test(b, n, p_0, alternative = "greater")
my_test

# Access just the p-value:
my_test$p.value

In [ ]:
# Manual p-value using pbinom (equivalent to binom.test for right-sided alternative)
# P(B >= b) = P(B > b-1) with lower.tail = FALSE
pbinom(b - 1, n, p_0, lower.tail = FALSE)

In [ ]:
# Asymptotic (normal approximation) binomial test
# Standardized statistic B* ~ N(0,1) under H0
B_star <- (b - n * p_0) / sqrt(n * p_0 * (1 - p_0))

# Critical value for right-sided test at alpha = 0.05
qnorm(0.95, mean = 0, sd = 1)

# p-value from the asymptotic test
pnorm(B_star, mean = 0, sd = 1, lower.tail = FALSE)

### 2.1 Finding the Critical Region

For the exact test, the critical region is the set of values of $B$ that lead to rejection. With a **left-sided** alternative at $\alpha = 0.05$:

In [ ]:
# Left-sided exact binomial critical region
# H0: p = 0.3, H1: p < 0.3, n = 50
n   <- 50
p_0 <- 0.3

# qbinom with left-sided alternative returns the value ONE ABOVE the critical boundary
qb <- qbinom(0.05, n, p_0, lower.tail = TRUE)
qb

# Verify: P(B <= qb-1) < 0.05 and P(B <= qb) > 0.05
pbinom(qb - 1, n, p_0, lower.tail = TRUE)
pbinom(qb,     n, p_0, lower.tail = TRUE)

In [ ]:
# Asymptotic critical region using normal quantile
C_krit <- qnorm(0.05) * sqrt(n * p_0 * (1 - p_0)) + n * p_0
floor(C_krit)   # floor() rounds down

---

## 3. Power Analysis for the Binomial Test

The **power function** $\beta(p)$ gives the probability of rejecting $H_0$ when the true probability of success is $p$. It is defined using the critical region found above.

In [ ]:
# Power function for left-sided binomial test (H0: p=0.3, critical region: B <= 9)
# n = 50, critical boundary at qb - 1 = 9
n   <- 50
p_0 <- 0.3
qb  <- qbinom(0.05, n, p_0, lower.tail = TRUE)

power <- function(p) { pbinom(qb - 1, n, p, lower.tail = TRUE) }

# Plot power as a function of p
plot(power, 0, 0.5,
     xlab = "True probability p",
     ylab = "Power  beta(p)",
     main = "Power Function — Exact Binomial Test")

# Power at a specific alternative (H1: p = 0.25)
power(0.25)

# Type II error probability (beta): probability of failing to reject H0 when H1 is true
cat("Type II error at p=0.25:", 1 - power(0.25), "\n")

### 3.1 Sample Size Determination

How many trials are needed so that the test has:
- Type I error probability $\leq 5\%$
- Type II error probability $\leq 2.5\%$ when $H_1: p = p_1$

In [ ]:
# Required sample size as a function of the alternative p1
# H0: p = 0.3, left-sided alternative, alpha = 0.05, beta = 0.025
p_0 <- 0.3

m <- function(p) {
  ceiling(
    ((qnorm(0.975) * sqrt(p * (1 - p)) - sqrt(p_0 * (1 - p_0)) * qnorm(0.05)) /
     (p_0 - p))^2
  )
}

# Required n when the true p is 0.25
cat("Required n for p1=0.25:", m(0.25), "\n")

# Required n when the true p is 0.20
cat("Required n for p1=0.20:", m(0.20), "\n")

---

## 4. Sign Test

The **sign test** is the simplest nonparametric test for paired data. It tests whether the median difference between paired observations is zero.

**Idea:** Let $D_i = Y_i - X_i$ (after minus before). Under $H_0$, each $D_i$ is equally likely to be positive or negative, so $B = \sum \mathbb{1}[D_i > 0] \sim \text{Binomial}(n, 0.5)$.

The sign test is **identical** to a binomial test with $p_0 = 0.5$.

In [ ]:
# Load weight data: patients measured before and after a diet intervention
# Columns: weight_before, weight_after
data <- read.table("../Assets/data/r_statistics/Weight.txt",
                   header = TRUE, sep = " ", dec = ".")

weight_before <- data[[1]]
weight_after  <- data[[2]]

head(data)

In [ ]:
# H0: median difference = 0, H1: median difference < 0 (weight decreases after diet)
b <- sum(weight_after > weight_before)   # count of positive differences
n <- length(weight_before)

cat("Positive differences (weight increased):", b, "of", n, "\n")

# Exact sign test = exact binomial test with p0 = 0.5
binom.test(b, n, p = 0.5, alternative = "less")

In [ ]:
# Equivalent manual calculation using pbinom
pbinom(b, n, 0.5, lower.tail = TRUE)

# Asymptotic sign test: standardized B*
B_star <- (b - n * 0.5) / sqrt(n * 0.5 * 0.5)
cat("Asymptotic statistic B*:", B_star, "\n")

# Normal critical value for left-sided test at alpha = 0.05
qnorm(0.05, mean = 0, sd = 1, lower.tail = TRUE)

# Asymptotic p-value
pnorm(B_star, mean = 0, sd = 1, lower.tail = TRUE)

---

## 5. Wilcoxon Signed-Rank Test

The **Wilcoxon signed-rank test** is more powerful than the sign test because it uses both the sign *and* the magnitude of each difference $D_i = Y_i - X_i$.

**Procedure:**
1. Compute $D_i = Y_i - X_i$ for each pair; discard ties ($D_i = 0$).
2. Rank the absolute differences $|D_i|$ from 1 to *n*.
3. Compute $W^+ = $ sum of ranks where $D_i > 0$, $W^- = $ sum of ranks where $D_i < 0$.
4. The test statistic is $V = \min(W^+, W^-)$.

**Assumption:** The differences $D_i$ are symmetrically distributed around a common median $\theta$.

**Null hypothesis:** $\theta = 0$.

In [ ]:
# Install exactRankTests for exact p-values with ties or small n
# install.packages("exactRankTests")
library(exactRankTests)

# Wilcoxon signed-rank test on weight data (paired = TRUE)
wilcox.test(weight_after, weight_before,
            paired = TRUE, alternative = "less", exact = TRUE)

In [ ]:
# wilcox.exact from exactRankTests — preferred for exact p-values with potential ties
wilcox.exact(weight_after, weight_before,
             paired = TRUE, alternative = "less", exact = TRUE)

### 5.1 Manual Step-by-Step Wilcoxon Sign-Rank Calculation

The following recreates the Wilcoxon signed-rank test by hand using the journey time example from Practicum 2: minutes of commute before and after a new road was opened.

In [ ]:
# Journey times (minutes) before and after road opening
x <- c(125.3, 101.0, 117.2, 133.7,  96.4, 124.5, 118.7, 106.2, 116.3, 120.2, 125.0, 128.8)
y <- c(127.3, 120.2, 126.2, 125.4, 115.1, 118.5, 135.5, 118.2, 122.9, 120.1, 120.8, 130.7)

# Built-in test for comparison
wilcox.test(x, y, paired = TRUE)

In [ ]:
# Manual calculation
df <- data.frame(x = x, y = y)
df$diff <- with(df, y - x)

# Assign ranks to |diff| (sorted by absolute value)
df[order(abs(df$diff)), 'rk'] <- 1:nrow(df)

# Signed ranks
df$signrk <- df$rk * sign(df$diff)
df

# W+ = sum of positive signed ranks, W- = |sum of negative signed ranks|
r <- df$signrk
W_plus  <- sum(r[r > 0])
W_minus <- sum(-r[r < 0])
cat("W+ =", W_plus, " W- =", W_minus, "\n")

# Asymptotic approximation for V = min(W+, W-)
n_eff <- nrow(df)
V <- min(W_plus, W_minus)
z <- (V - 0) / sqrt(n_eff * (n_eff + 1) * (2 * n_eff + 1) / 6)
cat("Two-sided asymptotic p-value:", 2 * pnorm(-abs(z)), "\n")

---

## 6. Wilcoxon Rank-Sum Test (Mann-Whitney U)

The **Wilcoxon rank-sum test** (equivalent to Mann-Whitney U) compares the location of two **independent** samples without assuming normality.

**Null hypothesis:** The two populations have the same distribution (or equivalently, $P(X > Y) = 0.5$).

Use `paired = FALSE` for independent samples.

In [ ]:
# Load blood pressure data for female and male patients
female_data <- read.table("../Assets/data/r_statistics/female.csv",
                          header = TRUE, sep = ";", dec = ",")
male_data   <- read.table("../Assets/data/r_statistics/male.csv",
                          header = TRUE, sep = ";", dec = ",")

female <- female_data[[1]]
male   <- male_data[[1]]

cat("Female n =", length(female), "  Male n =", length(male), "\n")
cat("Female median:", median(female), "  Male median:", median(male), "\n")

In [ ]:
# Two independent samples (paired = FALSE)
# H1: female values are LESS than male values
wilcox.exact(female, male,
             paired = FALSE, alternative = "less", exact = TRUE)

In [ ]:
# Two-sided test
wilcox.test(female, male,
            paired = FALSE, alternative = "two.sided", exact = FALSE)

### 6.1 Manual Rank-Sum Calculation

This reproduces the Mann-Whitney U statistic step by step using two treatment groups from Practicum 2.

In [ ]:
# Two independent samples: antibody concentrations (arbitrary units)
x_mw <- c(84, 128, 102, 117,  71,  75)
y_mw <- c(140,  95, 106,  99, 100, 135, 155, 158)

# Built-in test
wilcox.test(x_mw, y_mw)

# Manual U statistic
df2 <- rbind(data.frame(value = x_mw, sample = 1),
             data.frame(value = y_mw, sample = 2))
df2[order(df2$value), 'rk'] <- 1:nrow(df2)

# Rank sums per group
R <- with(df2, aggregate(list(rk = rk), by = list(sample = sample), FUN = sum))$rk
N <- with(df2, aggregate(list(rk = rk), by = list(sample = sample), FUN = length))$rk

U <- min(R - N * (N + 1) / 2)
cat("U statistic:", U, "\n")

# Asymptotic p-value (two-sided)
z_mw <- (U - N[1] * N[2] / 2) / sqrt(N[1] * N[2] * (N[1] + N[2] + 1) / 12)
cat("Two-sided p-value (asymptotic):", 2 * pnorm(z_mw), "\n")

---

## 7. Hodges-Lehmann Estimation

The **Hodges-Lehmann estimator** is a robust estimate of location associated with the Wilcoxon tests.

- For **paired data** (signed-rank test): $\hat{\theta} = \text{median}\left(\frac{D_i + D_j}{2}\right)$ for all $i \leq j$ — the median of all pairwise averages of the differences.
- For **two independent samples** (rank-sum test): $\hat{\Delta} = \text{median}(X_i - Y_j)$ for all $i, j$ — the median of all pairwise differences.

Add `conf.int = TRUE` to `wilcox.test` or `wilcox.exact` to obtain both the estimate and a confidence interval.

In [ ]:
# Hodges-Lehmann estimate for paired weight data
wilcox.exact(weight_after, weight_before,
             paired = TRUE, alternative = "less",
             exact = TRUE, conf.int = TRUE)

In [ ]:
# Hodges-Lehmann estimate for two independent groups
wilcox.exact(female, male,
             paired = FALSE, alternative = "less",
             exact = TRUE, conf.int = TRUE)

In [ ]:
# Robustness demonstration: introduce an outlier in the female data
female_2 <- female
f_length <- length(female)
female_2[f_length] <- female_2[f_length] * 10   # inflate last value 10-fold

cat("Without outlier — H-L estimate:\n")
print(wilcox.exact(female, male,   paired=FALSE, alt="less", conf.int=TRUE)$estimate)

cat("\nWith outlier — H-L estimate:\n")
print(wilcox.exact(female_2, male, paired=FALSE, alt="less", conf.int=TRUE)$estimate)

cat("\nMean without outlier:", mean(female), "  Mean with outlier:", mean(female_2), "\n")

The Hodges-Lehmann estimate is barely affected by the 10-fold outlier, whereas the sample mean changes substantially. This robustness is a key advantage of rank-based methods.

---

## 8. Kruskal-Wallis Test

The **Kruskal-Wallis test** is the nonparametric analogue of one-way ANOVA. It tests whether $k \geq 2$ independent groups have the same distribution.

**Null hypothesis:** All groups come from the same population.

The test statistic ranks all $N$ observations jointly, then computes a weighted sum of rank-sum deviations from the grand mean.

Under $H_0$ (for large $n$): $H \sim \chi^2_{k-1}$.

In [ ]:
# Example 1: Crop yield under three soil types
harvest_data <- read.table("../Assets/data/r_statistics/harvest_new.txt",
                           header = TRUE, sep = " ", dec = ".")

type_1 <- harvest_data[[1]]
type_2 <- harvest_data[[2]]
type_3 <- harvest_data[[3]]

cat("Medians by soil type:",
    median(type_1), median(type_2), median(type_3), "\n")

# Kruskal-Wallis test
kruskal.test(list(type_1, type_2, type_3))

In [ ]:
# Example 2: Test scores by occupation category (teacher, admin, personal)
school_data <- read.table("../Assets/data/r_statistics/school.txt",
                          header = TRUE, sep = " ", dec = ".")

teacher  <- school_data[[1]]
admin    <- school_data[[2]]
personal <- school_data[[3]]

kruskal.test(list(teacher, admin, personal))

### 8.1 Post-Hoc Dunn Test for Multiple Comparisons

The Kruskal-Wallis test tells us whether *any* groups differ, but not *which* pairs differ. The **Dunn test** provides asymptotic pairwise comparisons.

In [ ]:
# install.packages("PMCMR")
library(PMCMR)

# Asymptotic Dunn test — no p-value adjustment ("none") for illustration
posthoc.kruskal.dunn.test(school_data, p.adjust.method = "none")

In [ ]:
# With Bonferroni correction for multiple comparisons
posthoc.kruskal.dunn.test(school_data, p.adjust.method = "bonferroni")

---

## 9. Pressure Data: Putting It Together

The `Pressure.txt` file contains blood pressure measurements. We demonstrate a complete analysis workflow: load data, visualize, choose appropriate test, and report.

In [ ]:
# Load blood pressure data
pressure_data <- read.table("../Assets/data/r_statistics/Pressure.txt",
                            header = TRUE, sep = " ", dec = ".")
head(pressure_data)
str(pressure_data)

In [ ]:
# Visual check before choosing a test
par(mfrow = c(1, 2))
boxplot(pressure_data, main = "Blood Pressure by Group",
        col = c("lightblue", "lightcoral"), ylab = "mmHg")

# Histograms of each group
hist(pressure_data[[1]], main = "Group 1", xlab = "mmHg", col = "lightblue", breaks = 8)
par(mfrow = c(1, 1))

---

## 10. Exercises

**Exercise 1 — Exact Binomial Test**  
A clinical trial reports that 42 of 120 patients responded to a new drug. The historical response rate for the standard treatment is 40%. Test whether the new drug differs from the standard at $\alpha = 0.05$ (two-sided). Compute both the exact p-value using `binom.test` and the asymptotic p-value manually.

**Exercise 2 — Power Function**  
Using the binomial test with $H_0: p = 0.4$ and left-sided alternative, $n = 120$, $\alpha = 0.05$:  
a) Find the critical region boundary.  
b) Plot the power function $\beta(p)$ for $p \in [0, 0.4]$.  
c) What is the Type II error rate if the true $p = 0.30$?

**Exercise 3 — Sign Test vs Wilcoxon Signed-Rank**  
Load `Weight.txt`. Perform both the sign test and the Wilcoxon signed-rank test for the hypothesis that the diet intervention reduces weight ($H_1$: after < before, left-sided alternative). Compare p-values and explain why they differ.

**Exercise 4 — Wilcoxon Rank-Sum Test**  
Load `female.csv` and `male.csv`. Perform a two-sided Wilcoxon rank-sum test. Then compute the Hodges-Lehmann estimate with a 95% confidence interval. Interpret the result.

**Exercise 5 — Kruskal-Wallis and Post-Hoc**  
Load `harvest_new.txt`. Run the Kruskal-Wallis test. If the result is significant at $\alpha = 0.05$, identify which pairs of soil types differ using the Dunn test with Bonferroni correction.

**Exercise 6 — Sample Size**  
A researcher wants to test $H_0: p = 0.5$ against $H_1: p < 0.5$ with $\alpha = 0.05$ and power $\geq 0.90$ when the true $p = 0.35$. Using the formula from Section 3.1, calculate the required number of observations.

In [ ]:
# Exercise 1 workspace


In [ ]:
# Exercise 2 workspace


In [ ]:
# Exercise 3 workspace


In [ ]:
# Exercise 4 workspace


In [ ]:
# Exercise 5 workspace


In [ ]:
# Exercise 6 workspace


---

## Summary

| Test | Function | When to use |
|------|----------|-------------|
| Exact binomial | `binom.test` | Binary outcome, known n, test p against p0 |
| Sign test | `binom.test(b, n, 0.5)` | Paired data, minimal assumptions, count only sign |
| Wilcoxon signed-rank | `wilcox.test(paired=TRUE)` | Paired data, symmetric differences, uses magnitudes |
| Wilcoxon rank-sum (Mann-Whitney U) | `wilcox.test(paired=FALSE)` | Two independent samples, non-normal or small n |
| Kruskal-Wallis | `kruskal.test` | 3+ independent groups, nonparametric |
| Dunn post-hoc | `posthoc.kruskal.dunn.test` | Pairwise comparison after Kruskal-Wallis |
| Hodges-Lehmann CI | `conf.int=TRUE` in `wilcox.test` | Robust location estimate with CI |

**Key decisions:**
- Paired vs independent: determines test type (`paired=TRUE/FALSE`)
- Small n or potential ties: use `wilcox.exact` from `exactRankTests`
- Post-hoc after Kruskal-Wallis: Dunn test with Bonferroni or BH correction

---[< Previous: Probability and Statistics with Python](../07_Probability_and_Statistics_Python/02_statistics_python.ipynb) | [Tier 0: Computational Foundations Overview](../README.md) | [Next: R Regression, Correlation, and Diagnostics >](02_r_regression_correlation_and_diagnostics.ipynb)---